In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import date
import holidays
from datetime import timedelta


c:\Users\Sultan Selin\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
## merged data from parquet file 

MERGED_PATH = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "merged_data.parquet"
)
data = pd.read_parquet(MERGED_PATH)
print(data["date"].dt.tz)
data.info()


Europe/Istanbul
<class 'pandas.DataFrame'>
RangeIndex: 84696 entries, 0 to 84695
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype                          
---  ------         --------------  -----                          
 0   date           84696 non-null  datetime64[us, Europe/Istanbul]
 1   consumption    84696 non-null  float64                        
 2   year           84696 non-null  int32                          
 3   month          84696 non-null  int32                          
 4   day            84696 non-null  int32                          
 5   weekday        84696 non-null  str                            
 6   national_temp  84696 non-null  float64                        
 7   hour           84696 non-null  int32                          
dtypes: datetime64[us, Europe/Istanbul](1), float64(2), int32(4), str(1)
memory usage: 4.5 MB


In [4]:
#Consumption Lag Features
data["consumption_lag_24"] = data["consumption"].shift(24)
data["consumption_lag_48"] = data["consumption"].shift(48)
data["consumption_lag_168"] = data["consumption"].shift(168)


#Consumption Rolling mean Features
data["consumption_rolling_mean_24"]  = data["consumption"].shift(24).rolling(24).mean()
data["consumption_rolling_mean_168"] = data["consumption"].shift(24).rolling(168).mean()
# shift(24) before rolling: forecast horizon is 24h, so no hour
# within that window is available at prediction time


In [5]:
print(data["date"].is_monotonic_increasing)
print(data.info())
data

True
<class 'pandas.DataFrame'>
RangeIndex: 84696 entries, 0 to 84695
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype                          
---  ------                        --------------  -----                          
 0   date                          84696 non-null  datetime64[us, Europe/Istanbul]
 1   consumption                   84696 non-null  float64                        
 2   year                          84696 non-null  int32                          
 3   month                         84696 non-null  int32                          
 4   day                           84696 non-null  int32                          
 5   weekday                       84696 non-null  str                            
 6   national_temp                 84696 non-null  float64                        
 7   hour                          84696 non-null  int32                          
 8   consumption_lag_24            84672 non-null  float64         

,date,consumption,year,month,day,weekday,national_temp,hour,consumption_lag_24,consumption_lag_48,consumption_lag_168,consumption_rolling_mean_24,consumption_rolling_mean_168
0,2017-01-01 00:00:00+03:00,27223.06,2017,1,1,Sunday,1.314326,0,NaN,NaN,NaN,NaN,NaN
1,2017-01-01 01:00:00+03:00,25825.90,2017,1,1,Sunday,0.719622,1,NaN,NaN,NaN,NaN,NaN
2,2017-01-01 02:00:00+03:00,24252.68,2017,1,1,Sunday,0.304513,2,NaN,NaN,NaN,NaN,NaN
3,2017-01-01 03:00:00+03:00,22915.47,2017,1,1,Sunday,-0.786601,3,NaN,NaN,NaN,NaN,NaN
4,2017-01-01 04:00:00+03:00,22356.99,2017,1,1,Sunday,-0.671492,4,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
84691,2026-08-30 19:00:00+03:00,43487.20,2026,8,30,Sunday,25.720893,19,46146.69,50946.09,48107.67,43362.090833,47861.601488
84692,2026-08-30 20:00:00+03:00,44306.44,2026,8,30,Sunday,24.377825,20,46276.80,50700.57,49124.39,43177.767083,47824.966548
84693,2026-08-30 21:00:00+03:00,43046.39,2026,8,30,Sunday,24.069101,21,44800.31,49012.85,47802.28,43002.244583,47790.170952
84694,2026-08-30 22:00:00+03:00,41560.60,2026,8,30,Sunday,23.356092,22,42736.17,47010.37,46263.88,42824.152917,47754.071429


In [6]:
# Calendar Related Features
data["is_weekend"] = data["weekday"].isin(["Saturday", "Sunday"]).astype(int)
data["DoW"] = data["date"].dt.dayofweek
data["DoY"] = data["date"].dt.dayofyear
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 84696 entries, 0 to 84695
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype                          
---  ------                        --------------  -----                          
 0   date                          84696 non-null  datetime64[us, Europe/Istanbul]
 1   consumption                   84696 non-null  float64                        
 2   year                          84696 non-null  int32                          
 3   month                         84696 non-null  int32                          
 4   day                           84696 non-null  int32                          
 5   weekday                       84696 non-null  str                            
 6   national_temp                 84696 non-null  float64                        
 7   hour                          84696 non-null  int32                          
 8   consumption_lag_24            84672 non-null  float64              

In [7]:
#Holiday Related Features
tr_holidays = holidays.TR(years=range(2017, 2027))

holiday_set = set(tr_holidays.keys()) ### gives dates of the holidays


def is_religious_holiday(date):
    day = date.date()
    name = tr_holidays.get(day,"")
    if "Kurban Bayramı" in name or "Ramazan Bayramı" in name:
        return True
    return False

data["is_religious_holiday"] = data["date"].apply(is_religious_holiday)



def is_arife(date):
    tomorrow = (date + timedelta(days=1)).date()
    today = date.date()
    tomorrow_name = tr_holidays.get(tomorrow,"")
    today_name = tr_holidays.get(today, "")
    if "Kurban Bayramı" in tomorrow_name or "Ramazan Bayramı" in tomorrow_name:
        if "Kurban Bayramı" not in today_name and "Ramazan Bayramı" not in today_name:
            return True
    return False

data["is_arife"] = data["date"].apply(is_arife)



def is_national_holiday(date):
    day = date.date()
    name = tr_holidays.get(day, "")
    if name == "":
        return False
    return "Yılbaşı"  in name or"Ulusal Egemenlik ve Çocuk Bayramı"  in name or "Emek ve Dayanışma Günü" in name or "Atatürk'ü Anma, Gençlik ve Spor Bayramı" in name or 'Demokrasi ve Millî Birlik Günü' in name or 'Zafer Bayramı' in name or 'Cumhuriyet Bayramı' in name

data["is_national_holiday"] = data["date"].apply(is_national_holiday)



def is_kopru(date):
    today = date.date()
    yesterday = (date - timedelta(days=1)).date()
    tomorrow = (date + timedelta(days=1)).date()
    
    today_is_workday = today not in holiday_set and today.weekday() < 5
    surrounded = (yesterday in holiday_set or yesterday.weekday() >= 5) and \
                 (tomorrow in holiday_set or tomorrow.weekday() >= 5)
    
    return today_is_workday and surrounded

data["is_kopru"] = data["date"].apply(is_kopru)


def bayram_day_index(date):
    if is_religious_holiday(date) == False:
        return 0
    elif  not is_religious_holiday(date-timedelta(days=1)):
        return 1
    elif is_religious_holiday(date-timedelta(days=2)) == False:
        return 2
    elif is_religious_holiday(date-timedelta(days=3)) == False:
        return 3
    elif is_religious_holiday(date-timedelta(days=4)) == False:
        return 4

data["bayram_day_index"] = data["date"].apply(bayram_day_index)




In [ ]:
#date(2026, 5, 27) in tr_holidays ----returns True or False
#tr_holidays.get('2026-05-27') ---------name
#tr_holidays['2026-05-27']   -----name
#data[data["is_national_holiday"] == True].groupby(data["date"].dt.year)["is_national_holiday"].sum() / 24 ----nb of holidays not true check with this
#data["bayram_day_index"].value_counts().sort_index() -------nb of bayram day index

True

In [8]:
# Temperature Features 

# HDD & CDD

T_base = 16

data["HDD"] = (T_base - data["national_temp"]).clip(lower=0) #clip(lower=0) works like max(0, negative)
data["CDD"] = (data["national_temp"] - T_base).clip(lower=0)

# Lag & Rolling Mean

data["temp_lag_24"]  = data["national_temp"].shift(24)
data["temp_lag_48"]  = data["national_temp"].shift(48)
data["temp_rolling_mean_24"] = data["national_temp"].shift(24).rolling(24).mean()


In [9]:
# Trend features
# --- Trend handling for tree-based models ---
# Tree models (LightGBM) cannot extrapolate beyond the value ranges
# they saw during training. A raw trend feature (time_idx, year) fails
# on val/test because those periods sit outside the training range.
#
# Fix: instead of feeding the model a trend feature, normalize the
# target itself. The model then learns "how far is this hour from its
# own recent baseline" instead of "what is the absolute consumption
# level" - a ratio that stays roughly stable even as the absolute
# level keeps rising over the years.

# Reference level: rolling 1-year average, using only past data
# (shift(24) keeps this consistent with the 24h forecast horizon)
data["yearly_baseline"] = data["consumption"].shift(24).rolling(8766).mean()

# Normalized target: ratio of actual consumption to its own trailing baseline
data["target_ratio"] = data["consumption"] / data["yearly_baseline"]

# NOTE: rolling(8766) = 1 year of hours, so the first year of the
# series will be NaN here. Consider a shorter window (e.g. 90 days)
# if losing a full year is too costly.

# At inference time, convert the model's ratio prediction back to
# absolute consumption:
# predicted_consumption = predicted_ratio * yearly_baseline
data

,date,consumption,year,month,day,weekday,national_temp,hour,consumption_lag_24,consumption_lag_48,...,is_national_holiday,is_kopru,bayram_day_index,HDD,CDD,temp_lag_24,temp_lag_48,temp_rolling_mean_24,yearly_baseline,target_ratio
0,2017-01-01 00:00:00+03:00,27223.06,2017,1,1,Sunday,1.314326,0,NaN,NaN,...,True,False,0,14.685674,0.000000,NaN,NaN,NaN,NaN,NaN
1,2017-01-01 01:00:00+03:00,25825.90,2017,1,1,Sunday,0.719622,1,NaN,NaN,...,True,False,0,15.280378,0.000000,NaN,NaN,NaN,NaN,NaN
2,2017-01-01 02:00:00+03:00,24252.68,2017,1,1,Sunday,0.304513,2,NaN,NaN,...,True,False,0,15.695487,0.000000,NaN,NaN,NaN,NaN,NaN
3,2017-01-01 03:00:00+03:00,22915.47,2017,1,1,Sunday,-0.786601,3,NaN,NaN,...,True,False,0,16.786601,0.000000,NaN,NaN,NaN,NaN,NaN
4,2017-01-01 04:00:00+03:00,22356.99,2017,1,1,Sunday,-0.671492,4,NaN,NaN,...,True,False,0,16.671492,0.000000,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84691,2026-08-30 19:00:00+03:00,43487.20,2026,8,30,Sunday,25.720893,19,46146.69,50946.09,...,True,False,0,0.000000,9.720893,23.592781,23.277993,21.967974,40589.056963,1.071402
84692,2026-08-30 20:00:00+03:00,44306.44,2026,8,30,Sunday,24.377825,20,46276.80,50700.57,...,True,False,0,0.000000,8.377825,22.749788,22.444577,21.980691,40588.471465,1.091602
84693,2026-08-30 21:00:00+03:00,43046.39,2026,8,30,Sunday,24.069101,21,44800.31,49012.85,...,True,False,0,0.000000,8.069101,22.351889,21.814356,22.003089,40587.751648,1.060576
84694,2026-08-30 22:00:00+03:00,41560.60,2026,8,30,Sunday,23.356092,22,42736.17,47010.37,...,True,False,0,0.000000,7.356092,21.627122,20.642315,22.044122,40586.758430,1.023994


In [ ]:
# Lighting features

from astral import LocationInfo
from astral.sun import sun

ankara = LocationInfo(latitude=39.93, longitude=32.86, timezone="Europe/Istanbul")
# unique days in the series (or use dayofyear for a lighter approach)

# Use a leap year (2020) as reference so day 366 exists
ref_year = 2020

sun_times = {}
for doy in range(1, 367):
    d = date(ref_year, 1, 1) + timedelta(days=doy - 1)
    s = sun(ankara.observer, date=d, tzinfo="Europe/Istanbul")
    sun_times[doy] = {
        "sunrise_hour": s["sunrise"].hour + s["sunrise"].minute / 60,
        "sunset_hour": s["sunset"].hour + s["sunset"].minute / 60,
    }

# Map to the main dataframe via dayofyear
data["sunrise_hour"] = data["DoY"].map(lambda d: sun_times[d]["sunrise_hour"])
data["sunset_hour"]  = data["DoY"].map(lambda d: sun_times[d]["sunset_hour"])

# Day length in hours
data["day_length"] = data["sunset_hour"] - data["sunrise_hour"]



# is_dark: 1 if before sunrise or after sunset
data["is_dark"] = (
    (data["hour"] < data["sunrise_hour"]) |
    (data["hour"] > data["sunset_hour"])
).astype(int)

# Hours to sunrise; NaN when daytime (0 would mean "right now")
# Evening darkness should point to tomorrow's sunrise

data["next_day_sunrise"] = data["DoY"].apply(lambda d: sun_times[d % 366 + 1]["sunrise_hour"])

# Hours to next sunrise; NaN when daytime
data["hour_to_sunrise"] = np.select(
    [data["hour"] < data["sunrise_hour"], data["hour"] > data["sunset_hour"]],
    [data["sunrise_hour"] - data["hour"], 24 - data["hour"] + data["next_day_sunrise"]],
    default=np.nan
)
#data = data.drop(columns= ["sunrise_hour","sunset_hour", "next_day_sunrise"])
data



,date,consumption,year,month,day,weekday,national_temp,hour,consumption_lag_24,consumption_lag_48,...,temp_lag_48,temp_rolling_mean_24,yearly_baseline,target_ratio,day_length,is_dark,hour_to_sunrise,sunrise_hour,sunset_hour,next_day_sunrise
0,2017-01-01 00:00:00+03:00,27223.06,2017,1,1,Sunday,1.314326,0,NaN,NaN,...,NaN,NaN,NaN,NaN,9.383333,1,8.166667,8.166667,17.55,8.166667
1,2017-01-01 01:00:00+03:00,25825.90,2017,1,1,Sunday,0.719622,1,NaN,NaN,...,NaN,NaN,NaN,NaN,9.383333,1,7.166667,8.166667,17.55,8.166667
2,2017-01-01 02:00:00+03:00,24252.68,2017,1,1,Sunday,0.304513,2,NaN,NaN,...,NaN,NaN,NaN,NaN,9.383333,1,6.166667,8.166667,17.55,8.166667
3,2017-01-01 03:00:00+03:00,22915.47,2017,1,1,Sunday,-0.786601,3,NaN,NaN,...,NaN,NaN,NaN,NaN,9.383333,1,5.166667,8.166667,17.55,8.166667
4,2017-01-01 04:00:00+03:00,22356.99,2017,1,1,Sunday,-0.671492,4,NaN,NaN,...,NaN,NaN,NaN,NaN,9.383333,1,4.166667,8.166667,17.55,8.166667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84691,2026-08-30 19:00:00+03:00,43487.20,2026,8,30,Sunday,25.720893,19,46146.69,50946.09,...,23.277993,21.967974,40589.056963,1.071402,13.166667,0,NaN,6.233333,19.40,6.233333
84692,2026-08-30 20:00:00+03:00,44306.44,2026,8,30,Sunday,24.377825,20,46276.80,50700.57,...,22.444577,21.980691,40588.471465,1.091602,13.166667,1,10.233333,6.233333,19.40,6.233333
84693,2026-08-30 21:00:00+03:00,43046.39,2026,8,30,Sunday,24.069101,21,44800.31,49012.85,...,21.814356,22.003089,40587.751648,1.060576,13.166667,1,9.233333,6.233333,19.40,6.233333
84694,2026-08-30 22:00:00+03:00,41560.60,2026,8,30,Sunday,23.356092,22,42736.17,47010.37,...,20.642315,22.044122,40586.758430,1.023994,13.166667,1,8.233333,6.233333,19.40,6.233333


In [ ]:
# Bool to int conversion
bool_cols = ["is_religious_holiday", "is_arife", "is_national_holiday", "is_kopru"]
data[bool_cols] = data[bool_cols].astype(int)
